# Boom Challenge - Prediction Notebook (Linear-Only)

Loads trained linear models for all 6 targets, transforms `test.csv`, and writes predictions
in the format of `prediction_submission_template.csv`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from scipy.special import expit


class LinearModel:
    """Linear-only model (must match training notebook class for joblib deserialization)."""
    def __init__(self, linear_features):
        self.linear_features = linear_features
        self.baseline = None

    def fit(self, X, y):
        self.baseline = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]).fit(X[self.linear_features], y)
        return self

    def predict(self, X):
        return self.baseline.predict(X[self.linear_features])

    def coefficients(self):
        return pd.Series(self.baseline.coef_, index=self.linear_features)

In [ ]:
data_dir = "../forward_prediction"
model_dir = "../models_linear"

train_X = pd.read_csv(f"{data_dir}/train.csv")
test_X = pd.read_csv(f"{data_dir}/test.csv")

print(f"Train: {train_X.shape}, Test: {test_X.shape}")

## Geometry Builders

In [ ]:
_EPS = 1e-6

def build_geometry_base(df):
    X = df.copy()
    X["L_char"] = (X["energy"] / (X["atmosphere"] * X["gravity"])) ** 0.25
    X["pi_strength"] = X["strength"] / (X["atmosphere"] * X["gravity"] * X["L_char"])
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    return X

def build_geometry_fines(df):
    X = build_geometry_base(df)
    X["D_fines"] = 40.0
    X["pi_threshold_fines"] = X["D_fines"] / X["L_char"]
    return X

def build_geometry_oversize(df):
    X = build_geometry_base(df)
    X["D_oversize"] = 120.0
    X["pi_threshold_oversize"] = X["D_oversize"] / X["L_char"]
    return X

## Feature Transforms per Target

Each target uses a slightly different feature engineering approach.

In [ ]:
_LOG_SOURCES_BASE = {
    "pi_strength": "log_pi_strength",
    "coupling": "log_coupling",
    "porosity": "log_porosity",
    "shape_factor": "log_shape",
    "energy": "log_energy",
    "strength": "log_strength",
    "gravity": "log_gravity",
    "atmosphere": "log_atmosphere",
}

_LOG_SOURCES_FINES = {**_LOG_SOURCES_BASE, "pi_threshold_fines": "log_pi_threshold_fines"}
_LOG_SOURCES_OVERSIZE = {**_LOG_SOURCES_BASE, "pi_threshold_oversize": "log_pi_threshold_oversize"}

In [ ]:
class ZLogBase:
    """Base z-log transformer: z-scores all log features."""
    def __init__(self, log_sources):
        self._log_sources = log_sources
        self._scalers = {}

    def _raw_logs(self, geom):
        logs = pd.DataFrame(index=geom.index)
        for source, log_col in self._log_sources.items():
            logs[log_col] = np.log(geom[source].clip(lower=_EPS))
        return logs

    def _z_transform(self, geom):
        X = geom.copy()
        logs = self._raw_logs(geom)
        for log_col in logs:
            raw_log = logs[[log_col]].to_numpy()
            X[f"{log_col}_raw"] = raw_log.ravel()
            X[log_col] = self._scalers[log_col].transform(raw_log).ravel()
        return X, logs

    def _fit_base(self, geom):
        logs = self._raw_logs(geom)
        for log_col in logs:
            self._scalers[log_col] = StandardScaler().fit(logs[[log_col]].to_numpy())
        return logs

In [ ]:
# --- P80: derived features computed from z-scored inputs, NOT independently z-scored ---

class ZLogP80(ZLogBase):
    def __init__(self):
        super().__init__(_LOG_SOURCES_BASE)

    def fit(self, geom):
        self._fit_base(geom)
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        X["log_pi_strength_sq"] = X["log_pi_strength"] ** 2
        X["log_pi_strength_cu"] = X["log_pi_strength"] ** 3
        X["log_energy_x_cos_angle"] = X["log_energy"] * X["cos_angle"]
        X["log_energy_x_sin_angle"] = X["log_energy"] * X["sin_angle"]
        X["log_coupling_sq"] = X["log_coupling"] ** 2
        X["log_porosity_sq"] = X["log_porosity"] ** 2
        return X

In [ ]:
# --- R95: squares z-scored from raw logs; interactions from z-scored inputs ---

class ZLogR95(ZLogBase):
    def __init__(self):
        super().__init__(_LOG_SOURCES_BASE)
        self._sq_sources = {"log_porosity": "log_porosity_sq", "log_coupling": "log_coupling_sq"}

    def fit(self, geom):
        logs = self._fit_base(geom)
        for log_col, sq_col in self._sq_sources.items():
            sq_vals = logs[log_col].to_numpy().reshape(-1, 1) ** 2
            self._scalers[sq_col] = StandardScaler().fit(sq_vals)
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        for log_col, sq_col in self._sq_sources.items():
            sq_vals = logs[log_col].to_numpy().reshape(-1, 1) ** 2
            X[sq_col] = self._scalers[sq_col].transform(sq_vals).ravel()
        X["log_energy_x_cos_angle"] = X["log_energy"] * X["cos_angle"]
        X["log_atmosphere_x_cos_angle"] = X["log_atmosphere"] * X["cos_angle"]
        X["log_strength_x_cos_angle"] = X["log_strength"] * X["cos_angle"]
        return X

In [ ]:
# --- Generic derived-source transformer (R50_fines, R50_oversize, fines_frac, oversize_frac) ---
# Derived features computed from z-scored inputs, then independently z-scored.

class ZLogDerived(ZLogBase):
    def __init__(self, log_sources, derived_sources):
        super().__init__(log_sources)
        self._derived_sources = derived_sources

    def _compute_derived(self, X, logs):
        raw = {}
        for name, spec in self._derived_sources.items():
            src, fn = spec
            if isinstance(src, tuple):
                a = logs[src[0]].to_numpy() if src[0] in logs.columns else X[src[0]].to_numpy()
                b = logs[src[1]].to_numpy() if src[1] in logs.columns else X[src[1]].to_numpy()
                raw[name] = fn(a, b)
            else:
                vals = logs[src].to_numpy() if src in logs.columns else X[src].to_numpy()
                raw[name] = fn(vals)
        return raw

    def fit(self, geom):
        logs = self._fit_base(geom)
        X_tmp = geom.copy()
        X_tmp["sin_angle"] = np.sin(X_tmp["angle_rad"])
        X_tmp["cos_angle"] = np.cos(X_tmp["angle_rad"])
        derived = self._compute_derived(X_tmp, logs)
        for name, vals in derived.items():
            self._scalers[name] = StandardScaler().fit(vals.reshape(-1, 1))
        return self

    def transform(self, geom):
        X, logs = self._z_transform(geom)
        derived = self._compute_derived(X, logs)
        for name, vals in derived.items():
            X[name] = self._scalers[name].transform(vals.reshape(-1, 1)).ravel()
        return X

In [ ]:
# ---------- Derived feature specs ----------

DERIVED_R50_FINES = {
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "cos_angle_sq": ("cos_angle", lambda x: x ** 2),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_energy_cu": ("log_energy", lambda x: x ** 3),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
    "log_strength_x_cos": (("log_strength", "cos_angle"), lambda a, b: a * b),
    "log_energy_x_gravity": (("log_energy", "log_gravity"), lambda a, b: a * b),
}

DERIVED_R50_OVERSIZE = {
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_energy_x_cos": (("log_energy", "cos_angle"), lambda a, b: a * b),
    "log_strength_x_cos": (("log_strength", "cos_angle"), lambda a, b: a * b),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
}

DERIVED_FINES_FRAC = {
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_pi_threshold_fines_sq": ("log_pi_threshold_fines", lambda x: x ** 2),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_energy_x_coupling": (("log_energy", "log_coupling"), lambda a, b: a * b),
    "log_strength_x_coupling": (("log_strength", "log_coupling"), lambda a, b: a * b),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
}

DERIVED_OVERSIZE_FRAC = {
    "log_pi_threshold_oversize_sq": ("log_pi_threshold_oversize", lambda x: x ** 2),
    "log_pi_threshold_oversize_cu": ("log_pi_threshold_oversize", lambda x: x ** 3),
    "log_pi_strength_sq": ("log_pi_strength", lambda x: x ** 2),
    "log_pi_strength_cu": ("log_pi_strength", lambda x: x ** 3),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_energy_x_strength": (("log_energy", "log_strength"), lambda a, b: a * b),
    "log_atm_x_strength": (("log_atmosphere", "log_strength"), lambda a, b: a * b),
    "log_strength_x_coupling": (("log_strength", "log_coupling"), lambda a, b: a * b),
    "log_porosity_x_strength": (("log_porosity", "log_strength"), lambda a, b: a * b),
    "log_atm_x_cos": (("log_atmosphere", "cos_angle"), lambda a, b: a * b),
}

In [ ]:
# ---------- Target configs ----------

TARGET_CONFIGS = {
    "P80": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogP80(),
        "model_file": "model_P80.joblib",
        "transform": "length",
    },
    "R95": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogR95(),
        "model_file": "model_R95.joblib",
        "transform": "length",
    },
    "R50_fines": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_BASE, DERIVED_R50_FINES),
        "model_file": "model_R50_fines.joblib",
        "transform": "length",
    },
    "R50_oversize": {
        "build_geometry": build_geometry_base,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_BASE, DERIVED_R50_OVERSIZE),
        "model_file": "model_R50_oversize.joblib",
        "transform": "length",
    },
    "fines_frac": {
        "build_geometry": build_geometry_fines,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_FINES, DERIVED_FINES_FRAC),
        "model_file": "model_fines_frac.joblib",
        "transform": "logit",
    },
    "oversize_frac": {
        "build_geometry": build_geometry_oversize,
        "z_log_factory": lambda: ZLogDerived(_LOG_SOURCES_OVERSIZE, DERIVED_OVERSIZE_FRAC),
        "model_file": "model_oversize_frac.joblib",
        "transform": "logit",
    },
}

In [ ]:
# ---------- Build features and predict for each target ----------

results = {}

for target_name, cfg in TARGET_CONFIGS.items():
    print(f"\n--- {target_name} ---")

    # Fit z_log on training data
    train_geom = cfg["build_geometry"](train_X)
    z_log = cfg["z_log_factory"]().fit(train_geom)

    # Transform test data
    test_geom = cfg["build_geometry"](test_X)
    X_test = z_log.transform(test_geom)

    # Load model and predict
    model = joblib.load(f"{model_dir}/{cfg['model_file']}")
    y_pred_transformed = model.predict(X_test)

    # Invert transform
    if cfg["transform"] == "length":
        y_pred = np.exp(y_pred_transformed) * X_test["L_char"].to_numpy()
    else:
        y_pred = expit(y_pred_transformed)

    results[target_name] = y_pred
    print(f"  Predictions: min={y_pred.min():.4f}, max={y_pred.max():.4f}, mean={y_pred.mean():.4f}")

In [ ]:
# ---------- Assemble and save submission ----------

submission = pd.DataFrame({
    "scenario_id": range(len(test_X)),
    "P80": results["P80"],
    "fines_frac": results["fines_frac"],
    "oversize_frac": results["oversize_frac"],
    "R95": results["R95"],
    "R50_fines": results["R50_fines"],
    "R50_oversize": results["R50_oversize"],
})

submission.to_csv(f"{data_dir}/prediction_submission.csv", index=False)
print(f"Saved prediction_submission.csv ({len(submission)} rows)")
submission.head(10)